In [1]:
# SparkSession
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Pertemuan6-ParquetETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/10/01 01:19:46 WARN Utils: Your hostname, nada resolves to a loopback address: 127.0.1.1; using 192.168.1.14 instead (on interface wlp1s0)
26/10/01 01:19:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/01 01:19:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [3]:
# Dataset dari 3 sumber sekaligus

import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


In [5]:
# 1. EXTRACT
# Membaca ketiga sumber data menjadi 3 Spark DataFrame terpisah
df_transaksi = spark.read.csv(
    "tugas6_transaksi.csv",
    header=True,
    inferSchema=True
)

df_produk = spark.read.json("tugas6_produk.json")

df_ulasan = spark.read.csv(
    "tugas6_ulasan.csv",
    header=True,
    inferSchema=True
)

#Tampilkan baris jumlah schema masing-masing
print("Jumlah baris data transaksi: ", df_transaksi.count())
df_transaksi.printSchema()

print("\nJumlah baris data produk: ", df_produk.count())
df_produk.printSchema()

print("\nJumlah baris data ulasan: ", df_ulasan.count())
df_ulasan.printSchema()

Jumlah baris data transaksi:  5000
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)


Jumlah baris data produk:  30
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)


Jumlah baris data ulasan:  3500
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [6]:
# 2. TRANSFORM - Penggabungan
# Menggabungkan transaksi dengan data ulasan
df_gabungan = df_transaksi.join(
    df_ulasan,
    on = "order_id",
    how = "left"
)

# Menggabungkan hasilnya dengan data produk
df_gabungan = df_gabungan.join(
    df_produk,
    on = "product_id",
    how = "inner"
)

# Menghitung total pendapatan
df_gabungan = df_gabungan.withColumn(
    "total_pendapatan",
    df_gabungan["unit_terjual"] * df_gabungan["harga"]
)

# Menampilkan beberapa data hasil penggabungan
df_gabungan.show(5)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|         1500000|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|          450000|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
o

In [8]:
# C. TRANSFORM - Penanganan Data Kosong & Pengayaan
# Menambahkan kolom ada_ulasan
from pyspark.sql.functions import col

# Menandai transaksi yang mewakili ulasan
df_gabungan = df_gabungan.withColumn(
    "ada_ulasan",
    col("rating").isNotNull()
)

# Mengisi rating yang kosong dengan 0
df_gabungan = df_gabungan.na.fill({"rating": 0})

# Menampilkan hasil
df_gabungan.select(
    "order_id",
    "rating",
    "ada_ulasan",
    "total_pendapatan"
).show(10)

+--------+------+----------+----------------+
|order_id|rating|ada_ulasan|total_pendapatan|
+--------+------+----------+----------------+
|     TX0|     1|      true|           50000|
|     TX1|     0|     false|         1500000|
|     TX2|     2|      true|          100000|
|     TX3|     5|      true|          300000|
|     TX4|     0|     false|          450000|
|     TX5|     5|      true|          225000|
|     TX6|     4|      true|          450000|
|     TX7|     0|     false|          150000|
|     TX8|     5|      true|          150000|
|     TX9|     0|     false|          250000|
+--------+------+----------+----------------+
only showing top 10 rows



In [10]:
# D. LOAD
# Menyimpan hasil ke HDFS
output_path = "hdfs://localhost:9000/user/nada/tugas6/hasil_etl"

df_gabungan.write \
    .mode("overwrite") \
    .partitionBy("kategori") \
    .parquet(output_path)

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Verifikasi_Hasil_ETL") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

output_path = "hdfs://localhost:9000/user/nada/tugas6/hasil_etl"

df_hasil = spark.read.parquet(output_path)

print("Jumlah baris setelah dibaca kembali:", df_hasil.count())

[Stage 1:>                                                          (0 + 3) / 3]

Jumlah baris setelah dibaca kembali: 5000


In [13]:
# E. Insight Akhir
# Menghitung presentase transaksi yang memiliki ulasan 
from pyspark.sql.functions import count, sum, col, round

insight = df_gabungan.groupBy("kategori").agg(
    count("*").alias("total_transaksi"),
    sum(col("ada_ulasan").cast("int")).alias("transaksi_dengan_ulasan")
)

insight = insight.withColumn(
    "presentase_ulasan",
    round(
        col("transaksi_dengan_ulasan") / col("total_transaksi") * 100,
        2
    )
)

insight.orderBy("presentase_ulasan").show()

# Presentase transaksi dengan ulasan terendah
kategori_terendah = insight.orderBy("presentase_ulasan").first()

print(
    "Kategori dengan presentase ulasan terendah: ",
    kategori_terendah["kategori"]
)

print(
    "Presentase ulasan: ",
    kategori_terendah["presentase_ulasan"], "%"
)

+------------+---------------+-----------------------+-----------------+
|    kategori|total_transaksi|transaksi_dengan_ulasan|presentase_ulasan|
+------------+---------------+-----------------------+-----------------+
|     Makanan|            536|                    369|            68.84|
|   Kesehatan|            961|                    662|            68.89|
|     Fashion|           1035|                    726|            70.14|
|Rumah Tangga|            815|                    575|            70.55|
|  Elektronik|           1653|                   1168|            70.66|
+------------+---------------+-----------------------+-----------------+

Kategori dengan presentase ulasan terendah:  Makanan
Presentase ulasan:  68.84 %
